# Module 02 — Data Cleaning (the unglamorous skill that pays)

Models are easy to swap. **Clean data is the moat.** Interviewers probe cleaning
hard because it separates people who ran a tutorial from people who've handled
real data. We'll clean `customers.csv` end-to-end and, crucially, **justify every
decision** — because "why did you drop those rows?" is a real interview question.

We cover, precisely:
1. Missing data — the three mechanisms (MCAR / MAR / MNAR) and what to *do*.
2. Wrong dtypes — numbers-as-strings, dates-as-strings.
3. Duplicates — exact and key-based.
4. Categorical noise — whitespace, casing, typos.
5. Outliers — detect with IQR & z-score, and decide keep/cap/drop.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/customers.csv")
print("shape:", df.shape)
df.info()

shape: (508, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    508 non-null    int64  
 1   age            488 non-null    float64
 2   income         485 non-null    float64
 3   city           508 non-null    object 
 4   plan           508 non-null    object 
 5   tenure_months  508 non-null    int64  
 6   monthly_spend  508 non-null    float64
 7   support_calls  508 non-null    int64  
 8   churn          508 non-null    int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 35.8+ KB


## 2.1 A cleaning philosophy (say this in interviews)

> "I never mutate the raw file. I keep the original, work on a copy, and record
> every transformation so the pipeline is **reproducible** and **auditable**.
> For each issue I ask: *what caused it*, *what's the risk of each fix*, and
> *does the fix leak information from the target?*"

That mindset alone makes you sound senior.

In [2]:
raw = df.copy()          # keep the untouched original
clean = df.copy()        # we transform this one

## 2.2 Duplicates — remove noise before you measure it

There are two kinds:
- **Exact duplicates**: entire rows repeated (data pipeline glitch).
- **Key duplicates**: same entity id appears twice (business-logic issue).

We planted 8 exact duplicates. But note: `customer_id` should be unique, so we
dedupe on that key too.

In [3]:
print("exact duplicate rows:", clean.duplicated().sum())
clean = clean.drop_duplicates()
print("after dropping exact dupes:", clean.shape)

print("duplicate customer_id:", clean.duplicated(subset='customer_id').sum())
clean = clean.drop_duplicates(subset="customer_id", keep="first")
print("after dedup on key:", clean.shape)

exact duplicate rows: 8
after dropping exact dupes: (500, 9)
duplicate customer_id: 0
after dedup on key: (500, 9)


**Takeaway:** dedupe *before* computing statistics; otherwise repeated rows bias
your means and inflate counts. Always state which key defines a unique row.

## 2.3 Categorical noise — standardize text keys

`city` has leading/trailing spaces and inconsistent casing (`NAIROBI`, ` nairobi `).
To software, `"Nairobi"` and `" NAIROBI "` are *different categories* — this
silently breaks group-bys, joins, and one-hot encoding.

In [4]:
print("BEFORE — raw unique cities:")
print(sorted(clean["city"].unique().tolist()))

clean["city"] = clean["city"].str.strip().str.title()

print("\nAFTER — standardized cities:")
print(sorted(clean["city"].unique().tolist()))

BEFORE — raw unique cities:
[' Kisumu ', ' Mombasa ', ' Nairobi ', ' Nakuru ', 'ELDORET', 'Eldoret', 'KISUMU', 'Kisumu', 'MOMBASA', 'Mombasa', 'NAIROBI', 'NAKURU', 'Nairobi', 'Nakuru']

AFTER — standardized cities:
['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']


**Takeaway:** `.str.strip().str.title()` (or `.lower()`) is a cheap fix that
prevents a whole class of silent bugs. For real typos (e.g. "Nairoby"), you'd map
them explicitly or use fuzzy matching — but never guess silently.

## 2.4 Wrong dtypes — make types match meaning

Check `info()`: are numbers stored as numbers, categories as `category`, dates as
`datetime`? Wrong types block math and waste memory. Here `income` became float
(because of the injected `NaN`), which is *correct*. We convert `plan` to an
ordered category to encode its natural order.

In [5]:
plan_type = pd.CategoricalDtype(categories=["Basic", "Standard", "Premium"],
                                ordered=True)
clean["plan"] = clean["plan"].astype(plan_type)
print(clean["plan"].dtype)
# Ordered categories allow comparisons:
print("rows with plan > Basic:", (clean["plan"] > "Basic").sum())

# Safe numeric coercion pattern (turns bad strings into NaN instead of crashing):
clean["income"] = pd.to_numeric(clean["income"], errors="coerce")
print("income dtype:", clean["income"].dtype)

category
rows with plan > Basic: 251
income dtype: float64


## 2.5 Missing data — the part everyone gets wrong

First, **quantify** it. Then diagnose the **mechanism**, because the mechanism
dictates the honest fix.

- **MCAR** (Missing Completely At Random): missingness unrelated to anything.
  Dropping is safe-ish but wasteful.
- **MAR** (Missing At Random): missingness depends on *other observed* columns.
  Imputation using those columns is defensible.
- **MNAR** (Missing Not At Random): missingness depends on the *unobserved value
  itself* (e.g. high earners hide income). Dangerous — may need a "missing"
  indicator or domain handling.

In [6]:
miss = clean.isna().sum()
miss_pct = (clean.isna().mean() * 100).round(1)
print(pd.DataFrame({"missing": miss, "pct": miss_pct})[miss > 0])

        missing  pct
age          20  4.0
income       23  4.6


In [7]:
# Diagnose: is income-missingness related to plan? (we built it as MAR on Premium)
tmp = clean.copy()
tmp["income_missing"] = tmp["income"].isna()
print("share of income missing, by plan:")
print(tmp.groupby("plan", observed=True)["income_missing"].mean().round(3))

share of income missing, by plan:
plan
Basic       0.000
Standard    0.000
Premium     0.242
Name: income_missing, dtype: float64


The missing rate is far higher for `Premium` → this is **MAR** (depends on the
observed `plan`). So a **group-wise median imputation by plan** is honest: we fill
using the most similar customers. We ALSO add a `income_was_missing` flag, because
the fact that it was missing can itself be predictive.

In [8]:
clean["income_was_missing"] = clean["income"].isna().astype(int)

# group-wise median (robust to the income outlier) imputation
clean["income"] = clean.groupby("plan", observed=True)["income"] \
                       .transform(lambda s: s.fillna(s.median()))

# age missingness looked random (MCAR-ish) -> simple median is fine
clean["age"] = clean["age"].fillna(clean["age"].median())

print("remaining missing:\n", clean.isna().sum()[clean.isna().sum() > 0]
      if clean.isna().sum().sum() else "none — all filled")

remaining missing:
 none — all filled


**Why median, not mean?** `income` is right-skewed and has an outlier; the mean is
dragged upward, the **median is robust**. Choosing median here shows you
understand your data, not just `fillna`.

**Leakage warning:** never impute using the *target* column, and (strictly) fit
imputation statistics on the **training set only**, then apply to test. In a real
pipeline we'd use `SimpleImputer` inside a `Pipeline` (Module 08) so test data
can't peek at training stats.

## 2.6 Outliers — detect, then *decide* (don't auto-delete)

Two standard detectors:
- **Z-score**: how many std devs from the mean. Assumes roughly normal; itself
  sensitive to outliers (mean/std get distorted).
- **IQR rule**: flag points below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`. Robust,
  distribution-free — the safer default.

In [9]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_bounds(clean["income"])
outliers = clean[(clean["income"] < low) | (clean["income"] > high)]
print(f"IQR bounds for income: [{low:,.0f}, {high:,.0f}]")
print("income outliers flagged:", len(outliers))
print(outliers[["customer_id", "plan", "income"]].head())

IQR bounds for income: [-9,879, 109,551]
income outliers flagged: 26
    customer_id      plan      income
0             1     Basic  5000000.00
6             7     Basic   164424.62
19           20     Basic   110687.75
53           54  Standard   135609.76
85           86     Basic   133072.35


Now the judgment call. Options:
1. **Keep** — if it's a real, meaningful extreme (a genuine high-net-worth client).
2. **Cap / winsorize** — clip to the bound, keeping the row but taming its leverage.
3. **Drop** — only if you're confident it's an error (e.g. impossible value).

Our planted value (5,000,000) is implausibly large vs the rest → likely a data
error. We'll **cap** it rather than drop, to keep the customer while limiting
distortion. We document the decision.

In [10]:
clean["income_capped"] = clean["income"].clip(lower=low, upper=high)
print("max income before cap:", clean["income"].max())
print("max income after  cap:", clean["income_capped"].max().round(0))

max income before cap: 5000000.0
max income after  cap: 109551.0


**Interview line:** *"I don't delete outliers reflexively. I check whether it's an
error or a real extreme, quantify its leverage, and choose keep/cap/drop with a
written rationale. For skewed money data I lean on IQR and robust statistics."*

## 2.7 Final validation — prove the data is clean

In [11]:
assert clean.duplicated().sum() == 0, "still have exact dupes"
assert clean["customer_id"].is_unique, "customer_id not unique"
assert clean.isna().sum().sum() == 0, "still have missing values"
assert clean["city"].str.strip().eq(clean["city"]).all(), "whitespace remains"

print("ALL CLEANING CHECKS PASSED ✅")
print("final shape:", clean.shape)
clean.to_csv("../data/customers_clean.csv", index=False)
print("saved data/customers_clean.csv")

ALL CLEANING CHECKS PASSED ✅
final shape: (500, 11)
saved data/customers_clean.csv


## 2.8 Mini-exercises

1. Re-diagnose `age` missingness: is it related to `plan` or `city`? Was median
   imputation justified?
2. Try **mean** imputation for income and compare the resulting distribution to
   the median version (plot histograms in Module 03). Which shifts more?
3. Change the IQR multiplier `k` from 1.5 to 3.0. How many outliers now? What does
   `k` control conceptually?
4. Write a one-paragraph "data cleaning report" for this dataset as if handing it
   to a teammate.

## Summary — the cleaning checklist you can recite

1. **Copy** the raw; never mutate it.
2. **Duplicates**: drop exact, then dedupe on the business key.
3. **Text/categoricals**: strip + normalize case; map real typos explicitly.
4. **Dtypes**: make types match meaning (numeric, category, datetime).
5. **Missing**: quantify → diagnose MCAR/MAR/MNAR → impute honestly (+ a missing
   flag); fit stats on train only (no leakage).
6. **Outliers**: detect with IQR/z-score, then **decide** keep/cap/drop with reasons.
7. **Validate** with assertions; save a clean artifact.

Next: **Module 03 — EDA & Visualization**, where clean data starts telling stories.